In [1]:
%load_ext autoreload
%autoreload 2

from model import LavaTubeFinder
from training_utils import *
import torch
from hirise_dataset import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Using cuda" if torch.cuda.is_available() else "Using cpu")


C:\Users\Utente\Desktop\lavatube_paper\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using cuda


In [2]:
lava_model = LavaTubeFinder(n_classes=4)
sample_image = torch.randn(1, 1, 224, 224)          # B, C, L, W
sample_sequence = torch.randn(1, 10, 1, 224, 224)   # B, S, L, W

lava_model(static_img=sample_image, thermal_seq=sample_sequence)

tensor([[ 0.4013, -0.0512, -0.3208,  0.1321]], grad_fn=<AddmmBackward0>)

In [3]:
image_data = pd.read_json("../data/final_data/image_dataset.json")
image_data

,iscrowd,image_id,bbox,segmentation,category_id,id,area,image_name,img_path
0,0,1,"[438, 449, 830, 799]","[[673, 458, 507, 581, 465, 694, 438, 821, 466,...",0,1,532485,ESP_011386_2065_RED_resized_0.5_lbl_0.tiff,data/DeepLandforms_dataset
1,0,2,"[523, 535, 1048, 1016]","[[935, 565, 1031, 543, 1145, 535, 1242, 570, 1...",1,2,806355,ESP_011386_2065_RED_resized_0.5_lbl_1.tiff,data/DeepLandforms_dataset
2,0,3,"[399, 407, 415, 399]","[[517, 411, 434, 473, 413, 529, 399, 593, 414,...",0,3,133052,ESP_011386_2065_RED_resized_1_lbl_0.tiff,data/DeepLandforms_dataset
3,0,4,"[508, 516, 524, 508]","[[714, 531, 762, 520, 819, 516, 867, 533, 903,...",1,4,201337,ESP_011386_2065_RED_resized_1_lbl_1.tiff,data/DeepLandforms_dataset
4,0,5,"[400, 404, 208, 200]","[[459, 406, 417, 437, 407, 465, 400, 497, 407,...",0,5,33296,ESP_011386_2065_RED_resized_2_lbl_0.tiff,data/DeepLandforms_dataset
...,...,...,...,...,...,...,...,...,...
2054,0,2055,[],[],3,2055,0,ESP_011325_1845_RED_y46393_x12426_s1218.tif,data/plain_terrain_dataset
2055,0,2056,[],[],3,2056,0,ESP_011325_1845_RED_y53860_x11991_s328.tif,data/plain_terrain_dataset
2056,0,2057,[],[],3,2057,0,ESP_011325_1845_RED_y34307_x6164_s2308.tif,data/plain_terrain_dataset
2057,0,2058,[],[],3,2058,0,ESP_011325_1845_RED_y33511_x12006_s1828.tif,data/plain_terrain_dataset


In [ ]:
from torchvision.transforms import transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.CenterCrop(1000)
])

image_dataset = Hirise_Dataset(image_data, root_dir="..", transform=transform)
train_dataloader, val_dataloader = create_dataloaders(image_dataset, batch_size=4)

In [ ]:
for batch_idx, (images, thermals, labels) in enumerate(train_dataloader):
    print(f"--- Batch {batch_idx + 1} ---")
    print(f"Images shape:   {images.shape}   | dtype: {images.dtype} | device: {images.device}")
    print(f"Thermals shape: {thermals.shape} | dtype: {thermals.dtype} | device: {thermals.device}")
    print(f"Labels shape:   {labels.shape}   | values: {labels.tolist()}")

    if batch_idx == 1:  # Stops after 2 batches (index 0 and 1)
        break

In [ ]:
history = train_model(
    model=lava_model,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    criterion=nn.CrossEntropyLoss(),
    optimizer=torch.optim.Adam(lava_model.parameters(), lr=0.0001),
    num_epochs=15,
    scheduler=None,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class_names = {0: "Vertical Pit/Skylight", 1: "Shallow Pit Chain", 2: "Sloped Pit", 3: "Plain Terrain"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lava_model.eval()

sample_idx = np.random.randint(len(val_dataloader.dataset))
image, thermal, true_label = val_dataloader.dataset[sample_idx]

with torch.no_grad():
    logits = lava_model(
        image.unsqueeze(0).to(device),
        thermal.unsqueeze(0).to(device),
    )
    predicted_label = torch.argmax(logits, dim=1).item()

plt.imshow(image.squeeze(0).cpu(), cmap="gray")
plt.title(
    f"Predicted: {class_names[predicted_label]} | True: {class_names[true_label]}"
)
plt.axis("off")
plt.show()